In [4]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

# Load datasets
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
   r"A:\Data\train", batch_size=32, image_size=(128, 128), label_mode='categorical'
)
val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    r"A:\Data\valid", batch_size=32, image_size=(128, 128), label_mode='categorical'
)


Found 4217 files belonging to 4 classes.
Found 2092 files belonging to 4 classes.


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model = Sequential([
    Input(shape=(128, 128, 3)),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, 3, activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [6]:
# Train the model
history = model.fit(train_dataset, validation_data=val_dataset, epochs=10)

# Save the model
model.save('trained_model.keras')


Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 38s 275ms/step - accuracy: 0.5068 - loss: 205.7302 - val_accuracy: 0.7706 - val_loss: 0.5353
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 271ms/step - accuracy: 0.6933 - loss: 0.6587 - val_accuracy: 0.8212 - val_loss: 0.4545
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 272ms/step - accuracy: 0.7402 - loss: 0.5999 - val_accuracy: 0.8260 - val_loss: 0.4098
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 273ms/step - accuracy: 0.7599 - loss: 0.5533 - val_accuracy: 0.8222 - val_loss: 0.4047
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 273ms/step - accuracy: 0.7655 - loss: 0.5539 - val_accuracy: 0.8399 - val_loss: 0.3767
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 274ms/step - accuracy: 0.7761 - loss: 0.5132 - val_accuracy: 0.8623 - val_loss: 0.3547
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 271ms/step - accuracy: 0.7962 - loss: 0.4743 - val_accuracy: 0.8466 - val_loss: 0.3759
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 275ms/step - accuracy: 0.7981 - loss:

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model = Sequential([
    Input(shape=(128, 128, 3)),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, 3, activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [8]:
# Convert to TensorFlow Lite model
def convert_to_tflite(keras_model):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    tflite_model = converter.convert()
    with open('model.tflite', 'wb') as f:
        f.write(tflite_model)

# Convert and save the model
convert_to_tflite(model)


INFO:tensorflow:Assets written to: C:\Users\Arman\AppData\Local\Temp\tmpfr83ik1l\assets


INFO:tensorflow:Assets written to: C:\Users\Arman\AppData\Local\Temp\tmpfr83ik1l\assets


Saved artifact at 'C:\Users\Arman\AppData\Local\Temp\tmpfr83ik1l'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_17')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2185958865936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2186018329648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2186018329120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2186018325424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2186018325776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2186018328768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2186018331232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2186018331936: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [9]:
import numpy as np
import tflite_runtime.interpreter as tflite
import cv2

# Load the TFLite model and allocate tensors
interpreter = tflite.Interpreter(model_path='model.tflite')
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Load and preprocess an image
img = cv2.imread('_0_4015166.jpg')
img_resized = cv2.resize(img, (128, 128))  # Ensure the image size matches the model's expected input size
input_data = np.expand_dims(img_resized, axis=0).astype(np.float32)

# Perform inference
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()
predictions = interpreter.get_tensor(output_details[0]['index'])
predicted_class = np.argmax(predictions)

# Display results
print(f"Predicted class: {predicted_class}")


ModuleNotFoundError: No module named 'tflite_runtime'